# 🏗️ Building a Complete RAG Pipeline

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Combine** all the RAG components we've learned into a single, working pipeline
2. **Build** each stage: document loading, chunking, embedding, vector storage, and retrieval
3. **Design** effective prompts that use retrieved context to answer questions
4. **Evaluate** how well your RAG system retrieves relevant information
5. **Identify** common failure modes and apply fixes

## Prerequisites

This notebook brings together everything from the previous four notebooks:

| Notebook | What We Learned | How We Use It Here |
|----------|----------------|--------------------|
| **01 - What is RAG** | The Retrieve-Augment-Generate pattern | Overall pipeline architecture |
| **02 - Text Embeddings** | Turning text into vectors with TF-IDF | Our embedding layer |
| **03 - Vector Stores** | Storing and searching embeddings efficiently | Our retrieval engine |
| **04 - Chunking Strategies** | Splitting documents into meaningful pieces | Our preprocessing step |

**Libraries used:** `numpy`, `matplotlib`, `math`, `re`, `collections` — no external dependencies!

---

## 🖼️ The Big Picture

Think of it like **building a car from parts you already know how to make**.

In the previous notebooks, we learned how to build the engine (embeddings), the wheels (vector store), the body (chunking), and the steering wheel (retrieval). Now we're going to **bolt everything together** into a working vehicle that can actually drive — meaning, answer real questions using real documents.

Imagine you're a librarian. When someone asks you a question:
1. You **know which shelves** to check (document loading & indexing)
2. You **quickly find the right pages** (vector search & retrieval)
3. You **read the relevant paragraphs** and craft a helpful answer (prompt engineering & generation)

That's exactly what our RAG pipeline does, step by step. Let's build it!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import math
import re
from collections import Counter, defaultdict

# --- Architecture Diagram: The Complete RAG Pipeline ---

fig, ax = plt.subplots(1, 1, figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('Complete RAG Pipeline Architecture', fontsize=18, fontweight='bold', pad=20)

# --- Row 1: Indexing Phase (top) ---
ax.text(7, 8.4, 'INDEXING PHASE (Offline / One-Time)', fontsize=13, fontweight='bold',
        ha='center', va='center', color='#1a237e',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#e8eaf6', edgecolor='#1a237e', linewidth=2))

indexing_boxes = [
    (0.5, 6.8, 2.4, 1.0, '#e3f2fd', '#1565c0', 'Documents\n(Raw Text)'),
    (3.8, 6.8, 2.4, 1.0, '#e8f5e9', '#2e7d32', 'Preprocessing\n& Chunking'),
    (7.1, 6.8, 2.4, 1.0, '#fff3e0', '#e65100', 'Embedding\nModel'),
    (10.4, 6.8, 2.8, 1.0, '#fce4ec', '#c62828', 'Vector Store\n(Index)'),
]

for (x, y, w, h, fc, ec, label) in indexing_boxes:
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=fc, edgecolor=ec, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, fontsize=10, fontweight='bold',
            ha='center', va='center', color=ec)

# Arrows for indexing phase
arrow_style = 'Simple,tail_width=4,head_width=12,head_length=8'
for (x1, x2) in [(2.9, 3.8), (6.2, 7.1), (9.5, 10.4)]:
    ax.annotate('', xy=(x2, 7.3), xytext=(x1, 7.3),
                arrowprops=dict(arrowstyle='->', color='#37474f', lw=2.5))

# --- Row 2: Query Phase (bottom) ---
ax.text(7, 5.0, 'QUERY PHASE (Online / Every Question)', fontsize=13, fontweight='bold',
        ha='center', va='center', color='#b71c1c',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#ffebee', edgecolor='#b71c1c', linewidth=2))

query_boxes = [
    (0.5, 2.8, 2.4, 1.0, '#e3f2fd', '#1565c0', 'User\nQuestion'),
    (3.8, 2.8, 2.4, 1.0, '#fff3e0', '#e65100', 'Embed\nQuery'),
    (7.1, 2.8, 2.4, 1.0, '#fce4ec', '#c62828', 'Vector\nSearch'),
    (10.4, 2.8, 2.8, 1.0, '#e8f5e9', '#2e7d32', 'Retrieved\nChunks'),
]

for (x, y, w, h, fc, ec, label) in query_boxes:
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=fc, edgecolor=ec, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, fontsize=10, fontweight='bold',
            ha='center', va='center', color=ec)

# Arrows for query phase
for (x1, x2) in [(2.9, 3.8), (6.2, 7.1), (9.5, 10.4)]:
    ax.annotate('', xy=(x2, 3.3), xytext=(x1, 3.3),
                arrowprops=dict(arrowstyle='->', color='#37474f', lw=2.5))

# Bottom row: Prompt + Generate
prompt_box = FancyBboxPatch((3.5, 0.8), 3.0, 1.0, boxstyle='round,pad=0.1',
                            facecolor='#f3e5f5', edgecolor='#6a1b9a', linewidth=2)
ax.add_patch(prompt_box)
ax.text(5.0, 1.3, 'Build Prompt\n(Context + Question)', fontsize=10, fontweight='bold',
        ha='center', va='center', color='#6a1b9a')

answer_box = FancyBboxPatch((8.0, 0.8), 3.0, 1.0, boxstyle='round,pad=0.1',
                            facecolor='#e0f7fa', edgecolor='#00695c', linewidth=2)
ax.add_patch(answer_box)
ax.text(9.5, 1.3, 'Generate\nAnswer', fontsize=14, fontweight='bold',
        ha='center', va='center', color='#00695c')

# Arrow from retrieved chunks down to prompt
ax.annotate('', xy=(9.5, 1.8), xytext=(11.8, 2.8),
            arrowprops=dict(arrowstyle='->', color='#37474f', lw=2.5, connectionstyle='arc3,rad=0.3'))

# Arrow from user question down to prompt
ax.annotate('', xy=(5.0, 1.8), xytext=(1.7, 2.8),
            arrowprops=dict(arrowstyle='->', color='#37474f', lw=2.5, connectionstyle='arc3,rad=-0.3'))

# Arrow from prompt to answer
ax.annotate('', xy=(8.0, 1.3), xytext=(6.5, 1.3),
            arrowprops=dict(arrowstyle='->', color='#37474f', lw=2.5))

# Dashed connection between vector store and vector search
ax.annotate('', xy=(8.3, 3.8), xytext=(11.8, 6.8),
            arrowprops=dict(arrowstyle='->', color='#c62828', lw=2, linestyle='dashed'))
ax.text(10.8, 5.3, 'same\nindex', fontsize=9, ha='center', va='center',
        color='#c62828', fontstyle='italic')

plt.tight_layout()
plt.show()

---

## 📦 Step 1: Document Loading & Preprocessing

Every RAG pipeline starts with **real documents**. Think of this step like going to the library and picking the books you want your system to know about.

We'll create five documents about space — each one packed with real facts. Then we'll clean them up so our system can work with them.

> **Analogy:** Imagine you're making a study guide for a test. First you gather all your textbooks and notes (loading), then you highlight the important parts and remove the doodles (preprocessing).

In [ ]:
# --- Step 1: Document Loading & Preprocessing ---

# Our space knowledge base: 5 real, factual documents
documents = {
    "solar_system": {
        "title": "The Solar System",
        "content": (
            "The Solar System consists of the Sun and everything that orbits around it, "
            "including eight planets, dwarf planets, moons, asteroids, and comets. The Sun, "
            "a medium-sized star, makes up about 99.86 percent of the total mass of the Solar System. "
            "It is composed primarily of hydrogen (about 73 percent) and helium (about 25 percent), "
            "with trace amounts of heavier elements like oxygen, carbon, neon, and iron. "
            "The eight planets, in order from the Sun, are Mercury, Venus, Earth, Mars, Jupiter, "
            "Saturn, Uranus, and Neptune. The four inner planets (Mercury, Venus, Earth, Mars) are "
            "rocky terrestrial planets, while the four outer planets are gas and ice giants. "
            "Mercury is the closest planet to the Sun at about 58 million kilometers, while Neptune "
            "orbits at roughly 4.5 billion kilometers away. Between Mars and Jupiter lies the "
            "asteroid belt, a region containing millions of rocky objects ranging from tiny pebbles "
            "to the dwarf planet Ceres, which is about 940 kilometers in diameter. Beyond Neptune "
            "lies the Kuiper Belt, home to dwarf planets like Pluto, Eris, Makemake, and Haumea. "
            "Pluto was reclassified from a planet to a dwarf planet in 2006 by the International "
            "Astronomical Union. Earth orbits the Sun at an average distance of about 150 million "
            "kilometers, a distance known as one astronomical unit (AU). Jupiter, the largest planet, "
            "has a diameter of about 139,820 kilometers and more than 90 known moons, including the "
            "four large Galilean moons: Io, Europa, Ganymede, and Callisto. Saturn is famous for its "
            "spectacular ring system made of ice and rock particles."
        ),
        "source": "space_encyclopedia"
    },
    "stars_galaxies": {
        "title": "Stars and Galaxies",
        "content": (
            "Stars are massive celestial bodies made of hot gas that produce energy through nuclear "
            "fusion in their cores. They come in many types depending on their size, temperature, "
            "and stage of life. Main sequence stars, like our Sun, fuse hydrogen into helium and "
            "spend most of their lives in this stable phase. Red dwarfs are the most common type of "
            "star in the universe, smaller and cooler than the Sun, with surface temperatures around "
            "2,500 to 3,500 Kelvin. They can burn for trillions of years because they consume fuel "
            "very slowly. White dwarfs are the remnants of stars that have exhausted their nuclear "
            "fuel; they are extremely dense, packing roughly the mass of the Sun into an object the "
            "size of Earth. Neutron stars form when massive stars explode in supernovae; they are "
            "incredibly dense, with a teaspoon of neutron star material weighing about 6 billion "
            "tons. The Milky Way is our home galaxy, a barred spiral galaxy containing an estimated "
            "100 to 400 billion stars. It has a diameter of about 100,000 light-years and the Sun "
            "is located about 26,000 light-years from the galactic center. Galaxies come in several "
            "types: spiral galaxies like the Milky Way and Andromeda have rotating arms of stars; "
            "elliptical galaxies are smooth and oval-shaped; and irregular galaxies have no defined "
            "shape, like the Large Magellanic Cloud. The Andromeda Galaxy is the nearest large galaxy "
            "to the Milky Way, located about 2.5 million light-years away. It is expected to collide "
            "with the Milky Way in about 4.5 billion years."
        ),
        "source": "space_encyclopedia"
    },
    "space_exploration": {
        "title": "Space Exploration",
        "content": (
            "Space exploration began in earnest during the 20th century. The Soviet Union launched "
            "Sputnik, the first artificial satellite, on October 4, 1957, marking the start of the "
            "space age. This was followed by the first human in space, Yuri Gagarin, who orbited "
            "Earth on April 12, 1961. The United States responded with the Apollo program, and on "
            "July 20, 1969, Apollo 11 astronauts Neil Armstrong and Buzz Aldrin became the first "
            "humans to walk on the Moon. Armstrong's famous words were 'That's one small step for "
            "man, one giant leap for mankind.' The International Space Station (ISS) has been "
            "continuously occupied since November 2000 and orbits Earth at an altitude of about "
            "408 kilometers. It serves as a laboratory for scientific research in microgravity. "
            "Mars exploration has been a major focus. NASA's Curiosity rover landed in Gale Crater "
            "on August 6, 2012, and has been studying Martian geology and climate. The Perseverance "
            "rover landed on February 18, 2021, in Jezero Crater, with a mission to search for "
            "signs of ancient microbial life and collect rock samples for future return to Earth. "
            "Perseverance also carried Ingenuity, the first helicopter to fly on another planet. "
            "The Hubble Space Telescope, launched in 1990, revolutionized astronomy with its deep "
            "field images showing thousands of galaxies. The James Webb Space Telescope (JWST), "
            "launched on December 25, 2021, is the most powerful space telescope ever built, "
            "observing the universe in infrared to study the earliest galaxies, star formation, "
            "and exoplanet atmospheres."
        ),
        "source": "space_history"
    },
    "black_holes": {
        "title": "Black Holes",
        "content": (
            "A black hole is a region of spacetime where gravity is so strong that nothing, not even "
            "light or other electromagnetic radiation, can escape once it crosses the event horizon. "
            "The event horizon is the boundary around a black hole beyond which no information can "
            "return to the outside universe. It is not a physical surface but rather a mathematical "
            "boundary defined by the Schwarzschild radius. The Schwarzschild radius is calculated "
            "using the formula r_s = 2GM/c^2, where G is the gravitational constant, M is the mass "
            "of the object, and c is the speed of light. For a black hole with the mass of our Sun, "
            "the Schwarzschild radius would be about 3 kilometers. There are several types of black "
            "holes. Stellar black holes form when massive stars (at least 20 to 25 times the mass "
            "of the Sun) collapse at the end of their lives. They typically have masses between 5 "
            "and 100 solar masses. Supermassive black holes exist at the centers of most galaxies "
            "and can have masses ranging from millions to billions of solar masses. Sagittarius A* "
            "is the supermassive black hole at the center of our Milky Way galaxy. It has a mass "
            "of about 4 million solar masses and is located approximately 26,000 light-years from "
            "Earth. In 2019, the Event Horizon Telescope collaboration produced the first direct "
            "image of a black hole, showing the supermassive black hole at the center of galaxy "
            "M87. Black holes can also emit powerful jets of particles and radiation from their "
            "poles, and matter spiraling into them forms an accretion disk that can be extremely "
            "luminous."
        ),
        "source": "astrophysics_textbook"
    },
    "the_moon": {
        "title": "The Moon",
        "content": (
            "The Moon is Earth's only natural satellite and the fifth largest moon in the Solar "
            "System. It formed approximately 4.5 billion years ago, most likely from debris "
            "created when a Mars-sized body called Theia collided with the early Earth. This is "
            "known as the giant impact hypothesis. The Moon has a diameter of about 3,474 kilometers "
            "and orbits Earth at an average distance of 384,400 kilometers. The Moon's gravitational "
            "pull is the primary cause of Earth's ocean tides. The Moon creates two tidal bulges on "
            "Earth: one on the side facing the Moon and one on the opposite side. Spring tides occur "
            "when the Sun, Moon, and Earth align, creating higher high tides and lower low tides. "
            "The Moon goes through eight distinct phases during its 29.5-day lunar cycle: new moon, "
            "waxing crescent, first quarter, waxing gibbous, full moon, waning gibbous, third quarter, "
            "and waning crescent. These phases result from the changing angle between the Sun, Moon, "
            "and Earth. Six Apollo missions successfully landed humans on the Moon between 1969 and "
            "1972: Apollo 11, 12, 14, 15, 16, and 17. A total of 12 astronauts walked on the lunar "
            "surface. The far side of the Moon, which always faces away from Earth due to tidal "
            "locking, was first photographed by the Soviet Luna 3 spacecraft in 1959. China's "
            "Chang'e 4 mission made the first landing on the far side in January 2019. Future "
            "missions include NASA's Artemis program, which aims to return humans to the Moon and "
            "establish a sustainable lunar presence as a stepping stone for Mars exploration."
        ),
        "source": "space_encyclopedia"
    }
}


def preprocess(text):
    """Clean and normalize text for RAG processing.
    
    Steps:
    1. Lowercase the text
    2. Normalize whitespace (collapse multiple spaces/newlines)
    3. Remove special characters but keep periods, commas, hyphens, apostrophes
    4. Strip leading/trailing whitespace
    """
    # Lowercase
    text = text.lower()
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    # Keep alphanumeric, spaces, basic punctuation
    text = re.sub(r"[^a-z0-9\s.,;:!?'\-/()]", '', text)
    # Strip
    text = text.strip()
    return text


# Preprocess all documents
for doc_id, doc in documents.items():
    doc['processed'] = preprocess(doc['content'])

# Print statistics
print("=" * 65)
print("DOCUMENT LOADING & PREPROCESSING STATS")
print("=" * 65)
print(f"{'Document':<25} {'Raw Chars':>10} {'Processed':>10} {'Words':>8}")
print("-" * 65)
total_words = 0
for doc_id, doc in documents.items():
    raw_len = len(doc['content'])
    proc_len = len(doc['processed'])
    word_count = len(doc['processed'].split())
    total_words += word_count
    print(f"{doc['title']:<25} {raw_len:>10,} {proc_len:>10,} {word_count:>8,}")
print("-" * 65)
print(f"{'TOTAL':<25} {'':>10} {'':>10} {total_words:>8,}")
print(f"\nDocuments loaded: {len(documents)}")
print(f"Sources: {set(d['source'] for d in documents.values())}")

---

## ✂️ Step 2: Chunking with Metadata

Now we need to break our documents into smaller pieces (chunks). This is like taking each book and making flashcards from it — each flashcard has a manageable amount of information.

**Why chunk?**
- Our search works better on focused passages than on entire documents
- We want to return *just the relevant part*, not a whole book

We'll use **sentence-aware chunking** with overlap (from Notebook 04) and attach **metadata** to each chunk so we always know where it came from.

In [ ]:
# --- Step 2: Chunking with Metadata ---

def chunk_documents(documents, chunk_size=200, overlap=50):
    """Split documents into overlapping chunks using sentence-aware splitting.
    
    Each chunk includes metadata: doc_id, title, source, chunk_index, char_range.
    
    Sentence-aware means we try not to cut in the middle of a sentence.
    Think of it like breaking a chocolate bar along the lines rather than
    snapping it at random.
    """
    all_chunks = []
    
    for doc_id, doc in documents.items():
        text = doc['processed']
        # Split into sentences
        sentences = re.split(r'(?<=[.!?])\s+', text)
        
        chunks = []
        current_chunk = ""
        current_start = 0
        sentence_buffer = []  # track sentences in current chunk
        
        for sentence in sentences:
            # If adding this sentence would exceed chunk_size
            if len(current_chunk) + len(sentence) + 1 > chunk_size and current_chunk:
                # Save the current chunk
                chunks.append({
                    'text': current_chunk.strip(),
                    'doc_id': doc_id,
                    'title': doc['title'],
                    'source': doc['source'],
                    'chunk_index': len(chunks),
                    'char_start': current_start,
                    'char_end': current_start + len(current_chunk.strip())
                })
                
                # Create overlap: take last portion of text
                # Use sentences for cleaner overlap
                overlap_text = ""
                for s in reversed(sentence_buffer):
                    if len(overlap_text) + len(s) + 1 <= overlap:
                        overlap_text = s + " " + overlap_text if overlap_text else s
                    else:
                        break
                
                current_start = current_start + len(current_chunk) - len(overlap_text)
                current_chunk = overlap_text + " " if overlap_text else ""
                sentence_buffer = [s for s in sentence_buffer if s in overlap_text] if overlap_text else []
            
            current_chunk += (" " if current_chunk else "") + sentence
            sentence_buffer.append(sentence)
        
        # Don't forget the last chunk!
        if current_chunk.strip():
            chunks.append({
                'text': current_chunk.strip(),
                'doc_id': doc_id,
                'title': doc['title'],
                'source': doc['source'],
                'chunk_index': len(chunks),
                'char_start': current_start,
                'char_end': current_start + len(current_chunk.strip())
            })
        
        all_chunks.extend(chunks)
    
    return all_chunks


# Chunk all documents
chunks = chunk_documents(documents, chunk_size=300, overlap=60)

# Print chunking stats
print("=" * 65)
print("CHUNKING STATISTICS")
print("=" * 65)

# Count chunks per document
chunks_per_doc = Counter(c['doc_id'] for c in chunks)
print(f"\nTotal chunks created: {len(chunks)}")
print(f"Chunk size target: 300 chars | Overlap: 60 chars\n")

print(f"{'Document':<25} {'Chunks':>8} {'Avg Chunk Len':>15}")
print("-" * 50)
for doc_id, count in chunks_per_doc.items():
    doc_chunks = [c for c in chunks if c['doc_id'] == doc_id]
    avg_len = np.mean([len(c['text']) for c in doc_chunks])
    print(f"{documents[doc_id]['title']:<25} {count:>8} {avg_len:>14.1f}")

# Show first 3 chunks as examples
print("\n" + "=" * 65)
print("SAMPLE CHUNKS (first 3)")
print("=" * 65)
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(f"  Doc: {chunk['title']} | Index: {chunk['chunk_index']}")
    print(f"  Chars: {chunk['char_start']}-{chunk['char_end']} ({len(chunk['text'])} chars)")
    print(f"  Text: {chunk['text'][:120]}...")

---

## 🧠 Step 3: Creating Embeddings (TF-IDF Based)

Now we need to turn our text chunks into **numbers** so the computer can compare them. This is where embeddings come in (from Notebook 02).

We'll use **TF-IDF** (Term Frequency - Inverse Document Frequency) — a simple but effective way to represent text as vectors.

> **Analogy:** Imagine each chunk is a recipe. TF-IDF figures out which ingredients are *special* to each recipe. If every recipe uses salt, salt isn't very helpful for telling recipes apart. But if only one recipe uses saffron, that's a great distinguishing ingredient!

- **TF (Term Frequency):** How often a word appears in this chunk
- **IDF (Inverse Document Frequency):** How rare/special a word is across ALL chunks

In [ ]:
# --- Step 3: TF-IDF Embeddings ---

class SimpleEmbedder:
    """A TF-IDF based text embedder built from scratch.
    
    This creates vector representations of text using term frequency
    and inverse document frequency.
    """
    
    def __init__(self, max_features=500):
        self.max_features = max_features
        self.vocabulary = {}     # word -> index mapping
        self.idf_values = {}     # word -> IDF score
        self.is_fitted = False
    
    def _tokenize(self, text):
        """Simple tokenization: lowercase, split on non-alpha, remove short words."""
        words = re.findall(r'[a-z][a-z0-9]+', text.lower())
        # Remove very common stopwords
        stopwords = {'the', 'is', 'at', 'of', 'on', 'and', 'a', 'an', 'in', 'to', 
                     'for', 'it', 'its', 'this', 'that', 'are', 'was', 'were', 'be',
                     'has', 'had', 'have', 'with', 'as', 'by', 'from', 'or', 'not',
                     'but', 'they', 'their', 'which', 'about', 'can', 'than', 'also',
                     'been', 'into', 'when', 'more', 'other', 'most', 'between'}
        return [w for w in words if w not in stopwords and len(w) > 1]
    
    def fit(self, texts):
        """Learn vocabulary and IDF values from a collection of texts."""
        n_docs = len(texts)
        
        # Count document frequency for each word
        doc_freq = Counter()
        all_word_counts = Counter()
        
        for text in texts:
            tokens = self._tokenize(text)
            unique_tokens = set(tokens)
            for token in unique_tokens:
                doc_freq[token] += 1
            for token in tokens:
                all_word_counts[token] += 1
        
        # Select top features by document frequency (but not TOO common)
        # Remove words that appear in >90% of docs (too common) or <2 docs (too rare)
        min_docs = min(2, n_docs)
        max_docs = max(int(0.9 * n_docs), 1)
        
        valid_words = {
            word: count for word, count in doc_freq.items()
            if min_docs <= count <= max_docs
        }
        
        # If we filtered too aggressively, fall back to all words with doc_freq >= 1
        if len(valid_words) < 20:
            valid_words = {word: count for word, count in doc_freq.items() if count >= 1}
        
        # Take top N by total frequency
        sorted_words = sorted(valid_words.keys(), 
                              key=lambda w: all_word_counts[w], reverse=True)
        selected = sorted_words[:self.max_features]
        
        # Build vocabulary mapping
        self.vocabulary = {word: idx for idx, word in enumerate(selected)}
        
        # Compute IDF: log(N / df) + 1 (smoothed)
        self.idf_values = {}
        for word in selected:
            df = doc_freq[word]
            self.idf_values[word] = math.log(n_docs / (df + 1)) + 1
        
        self.is_fitted = True
        print(f"Embedder fitted: {len(self.vocabulary)} features from {n_docs} documents")
        
        # Show top 10 most important words
        top_idf = sorted(self.idf_values.items(), key=lambda x: x[1], reverse=True)[:10]
        print(f"Top IDF words (most distinguishing): {[w for w, _ in top_idf]}")
        return self
    
    def embed(self, text):
        """Convert a single text into a TF-IDF vector."""
        if not self.is_fitted:
            raise ValueError("Embedder must be fitted before embedding!")
        
        tokens = self._tokenize(text)
        token_counts = Counter(tokens)
        n_tokens = len(tokens) if tokens else 1
        
        # Build TF-IDF vector
        vector = np.zeros(len(self.vocabulary))
        for word, idx in self.vocabulary.items():
            tf = token_counts.get(word, 0) / n_tokens  # Normalized TF
            idf = self.idf_values.get(word, 0)
            vector[idx] = tf * idf
        
        # L2 normalize
        norm = np.linalg.norm(vector)
        if norm > 0:
            vector = vector / norm
        
        return vector
    
    def embed_batch(self, texts):
        """Embed multiple texts at once."""
        return np.array([self.embed(text) for text in texts])


# Create and fit the embedder on all chunks
embedder = SimpleEmbedder(max_features=300)
chunk_texts = [c['text'] for c in chunks]
embedder.fit(chunk_texts)

# Embed all chunks
chunk_embeddings = embedder.embed_batch(chunk_texts)

print(f"\nEmbedding matrix shape: {chunk_embeddings.shape}")
print(f"  = {chunk_embeddings.shape[0]} chunks x {chunk_embeddings.shape[1]} dimensions")
print(f"\nNon-zero features per chunk (avg): {np.mean(np.count_nonzero(chunk_embeddings, axis=1)):.1f}")
print(f"Sparsity: {100 * (1 - np.count_nonzero(chunk_embeddings) / chunk_embeddings.size):.1f}%")

---

## 🗃️ Step 4: Vector Store

Now we need a place to **store** our embeddings and **search** through them quickly. This is the vector store (from Notebook 03).

> **Analogy:** The vector store is like the index at the back of a textbook. Instead of reading every page to find information about "Jupiter," you look up "Jupiter" in the index and it tells you exactly which pages to go to. Our vector store does the same thing, but with math instead of alphabetical order.

In [ ]:
# --- Step 4: Vector Store ---

class RAGVectorStore:
    """A simple vector store for RAG that stores chunks and their embeddings.
    
    Uses cosine similarity to find the most relevant chunks for a query.
    """
    
    def __init__(self):
        self.chunks = []         # List of chunk dictionaries
        self.embeddings = None   # numpy array of embeddings
        self.n_chunks = 0
    
    def add_chunks(self, chunks, embeddings):
        """Add chunks and their embeddings to the store."""
        self.chunks = chunks
        self.embeddings = embeddings
        self.n_chunks = len(chunks)
        print(f"Vector store loaded: {self.n_chunks} chunks, "
              f"embedding dim = {embeddings.shape[1]}")
    
    def _cosine_similarity(self, query_vec, doc_vecs):
        """Compute cosine similarity between a query and all documents.
        
        Since our vectors are already L2-normalized, this is just a dot product.
        """
        return np.dot(doc_vecs, query_vec)
    
    def search(self, query_embedding, top_k=5, min_score=0.0):
        """Find the top_k most similar chunks to the query.
        
        Args:
            query_embedding: The embedded query vector
            top_k: Number of results to return
            min_score: Minimum similarity score threshold
            
        Returns:
            List of (chunk, score) tuples, sorted by score descending
        """
        # Compute similarities
        scores = self._cosine_similarity(query_embedding, self.embeddings)
        
        # Get top-k indices
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        # Build results, filtering by min_score
        results = []
        for idx in top_indices:
            score = scores[idx]
            if score >= min_score:
                results.append((self.chunks[idx], float(score)))
        
        return results


# Create vector store and load our chunks
vector_store = RAGVectorStore()
vector_store.add_chunks(chunks, chunk_embeddings)

# Quick test: search for "planets in the solar system"
test_query = "planets in the solar system"
test_embedding = embedder.embed(test_query)
results = vector_store.search(test_embedding, top_k=3)

print(f"\nTest Query: '{test_query}'")
print("=" * 60)
for i, (chunk, score) in enumerate(results):
    print(f"\n  Result {i+1} (score: {score:.4f})")
    print(f"  Source: {chunk['title']} [chunk {chunk['chunk_index']}]")
    print(f"  Text: {chunk['text'][:150]}...")

---

## 🔍 Step 5: Query Pipeline

Now we connect everything together! The query pipeline takes a question, finds relevant chunks, builds a prompt, and generates an answer.

> **Analogy:** This is like the final assembly line at a car factory. The engine, wheels, body, and steering wheel are all ready — now we bolt them together and take the car for a test drive!

Since we don't have an actual LLM here, our "generation" step will format the retrieved context into a readable answer. In a real system, this is where you'd call GPT-4, Claude, or another language model.

In [ ]:
# --- Step 5: The Complete RAG Pipeline ---

class RAGPipeline:
    """A complete RAG pipeline that ties together all components.
    
    Components:
    - Embedder: converts text to vectors (TF-IDF)
    - Vector Store: stores and searches chunk embeddings
    - Prompt Builder: formats context + question into a prompt
    """
    
    def __init__(self, embedder, vector_store, top_k=3):
        self.embedder = embedder
        self.vector_store = vector_store
        self.top_k = top_k
    
    def retrieve(self, query, top_k=None):
        """Step 1: Retrieve relevant chunks for a query."""
        k = top_k or self.top_k
        query_embedding = self.embedder.embed(query)
        results = self.vector_store.search(query_embedding, top_k=k)
        return results
    
    def build_prompt(self, query, retrieved_chunks):
        """Step 2: Build a prompt with retrieved context."""
        # Format context from retrieved chunks
        context_parts = []
        for i, (chunk, score) in enumerate(retrieved_chunks):
            context_parts.append(
                f"[Source: {chunk['title']}, Relevance: {score:.2f}]\n"
                f"{chunk['text']}"
            )
        
        context = "\n\n".join(context_parts)
        
        prompt = (
            f"Answer the following question based ONLY on the provided context. "
            f"If the context doesn't contain enough information, say so.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {query}\n\n"
            f"Answer:"
        )
        return prompt
    
    def query(self, question, top_k=None, verbose=True):
        """Run the full RAG pipeline: retrieve -> build prompt -> display.
        
        In a real system, this would also call an LLM to generate the answer.
        Here we display the prompt that WOULD be sent to an LLM.
        """
        # Retrieve
        results = self.retrieve(question, top_k)
        
        # Build prompt
        prompt = self.build_prompt(question, results)
        
        if verbose:
            print("=" * 70)
            print(f"QUESTION: {question}")
            print("=" * 70)
            print(f"\nRetrieved {len(results)} chunks:")
            for i, (chunk, score) in enumerate(results):
                print(f"  {i+1}. [{chunk['title']}, chunk {chunk['chunk_index']}] "
                      f"score={score:.4f}")
            print(f"\n{'---' * 20}")
            print("GENERATED PROMPT (would be sent to LLM):")
            print(f"{'---' * 20}")
            print(prompt[:600])
            if len(prompt) > 600:
                print(f"... [truncated, total {len(prompt)} chars]")
            print()
        
        return {
            'question': question,
            'results': results,
            'prompt': prompt,
            'n_chunks': len(results)
        }


# Create the pipeline
rag = RAGPipeline(embedder, vector_store, top_k=3)

# Test with 4 space questions!
test_questions = [
    "How far is Neptune from the Sun?",
    "What is a neutron star and how dense is it?",
    "When did the Perseverance rover land on Mars?",
    "How does the Moon cause tides on Earth?"
]

for q in test_questions:
    rag.query(q)
    print()

---

## 📝 Step 6: Prompt Engineering for RAG

The prompt is the **instruction manual** you give to the LLM. A good prompt makes the difference between a helpful answer and a hallucinated mess.

> **Analogy:** Think of the LLM as a very smart student taking an open-book exam. The prompt is the exam question, and the retrieved chunks are the allowed reference pages. A well-written exam question guides the student to give a focused, accurate answer. A vague question leads to rambling or made-up answers.

We'll explore **three different prompt templates**, each suited for different tasks.

In [ ]:
# --- Step 6: Prompt Engineering Templates ---

def format_context(results):
    """Format retrieved chunks into a context string."""
    parts = []
    for i, (chunk, score) in enumerate(results):
        parts.append(f"[{i+1}] (Source: {chunk['title']})\n{chunk['text']}")
    return "\n\n".join(parts)


# Template 1: Factual Q&A (strict - only use provided context)
def prompt_factual_qa(question, context):
    return (
        "You are a helpful assistant that answers questions based ONLY on the "
        "provided context. Do not use any external knowledge. If the answer is "
        "not in the context, say 'I cannot find this information in the provided "
        "documents.'\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {question}\n\n"
        "ANSWER:"
    )


# Template 2: Summarization (synthesize info from multiple chunks)
def prompt_summarize(topic, context):
    return (
        "You are a helpful assistant. Using ONLY the information in the context "
        "below, provide a clear and concise summary of the topic. Organize the "
        "information logically and cite source numbers in brackets like [1].\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"TOPIC TO SUMMARIZE: {topic}\n\n"
        "SUMMARY:"
    )


# Template 3: Comparison (compare/contrast from context)
def prompt_compare(item_a, item_b, context):
    return (
        "You are a helpful assistant. Using ONLY the information in the context "
        "below, compare and contrast the two items. Create a structured comparison "
        "with similarities and differences. If information about one item is "
        "missing, note that.\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"COMPARE: {item_a} vs {item_b}\n\n"
        "COMPARISON:"
    )


# Demo each template type
print("=" * 70)
print("PROMPT TEMPLATE DEMOS")
print("=" * 70)

# Demo 1: Factual Q&A
q1 = "What is the Schwarzschild radius?"
r1 = rag.retrieve(q1, top_k=2)
ctx1 = format_context(r1)
prompt1 = prompt_factual_qa(q1, ctx1)
print("\n--- Template 1: Factual Q&A ---")
print(prompt1[:500])
print("..." if len(prompt1) > 500 else "")

# Demo 2: Summarization
q2 = "Mars exploration missions"
r2 = rag.retrieve(q2, top_k=3)
ctx2 = format_context(r2)
prompt2 = prompt_summarize(q2, ctx2)
print("\n--- Template 2: Summarization ---")
print(prompt2[:500])
print("..." if len(prompt2) > 500 else "")

# Demo 3: Comparison
r3_a = rag.retrieve("stellar black holes", top_k=2)
r3_b = rag.retrieve("neutron stars", top_k=2)
ctx3 = format_context(r3_a + r3_b)
prompt3 = prompt_compare("Stellar Black Holes", "Neutron Stars", ctx3)
print("\n--- Template 3: Comparison ---")
print(prompt3[:500])
print("..." if len(prompt3) > 500 else "")

print("\n" + "=" * 70)
print("KEY INSIGHT: Different question types need different prompt structures!")
print("The same retrieved context can produce very different answers depending")
print("on how you frame the prompt.")
print("=" * 70)

---

## 📊 Step 7: Evaluating RAG Quality

How do we know if our RAG system is actually **good**? We need metrics!

> **Analogy:** Imagine you're grading a student's open-book exam. You check two things:
> 1. **Precision:** Of the pages the student looked up, how many were actually relevant? (Did they waste time on irrelevant pages?)
> 2. **Recall:** Of all the relevant pages in the book, how many did the student find? (Did they miss important information?)

These are the two most important retrieval metrics:
- **Precision@k** = (relevant results in top k) / k
- **Recall@k** = (relevant results in top k) / (total relevant documents)

In [ ]:
# --- Step 7: RAG Evaluation ---

# Define test cases with expected relevant document IDs
test_cases = [
    {
        "query": "How many planets are in the solar system?",
        "relevant_docs": ["solar_system"],
        "description": "Basic factual question"
    },
    {
        "query": "What types of stars exist in the universe?",
        "relevant_docs": ["stars_galaxies"],
        "description": "Domain-specific vocabulary"
    },
    {
        "query": "Tell me about the first Moon landing and the Apollo missions",
        "relevant_docs": ["space_exploration", "the_moon"],
        "description": "Multi-document question"
    },
    {
        "query": "What is a black hole event horizon?",
        "relevant_docs": ["black_holes"],
        "description": "Technical concept"
    },
    {
        "query": "How were the Moon's craters formed and what causes tides?",
        "relevant_docs": ["the_moon"],
        "description": "Compound question"
    },
    {
        "query": "Mars rovers Curiosity and Perseverance",
        "relevant_docs": ["space_exploration"],
        "description": "Named entity search"
    },
]


def evaluate_retrieval(pipeline, test_cases, top_k=3):
    """Evaluate retrieval quality using Precision@k and Recall@k.
    
    For each test case:
    - Precision@k = (# relevant chunks in top k) / k
    - Recall@k = (# unique relevant docs found) / (# expected relevant docs)
    """
    results = []
    
    for tc in test_cases:
        query = tc['query']
        expected_docs = set(tc['relevant_docs'])
        
        # Retrieve
        retrieved = pipeline.retrieve(query, top_k=top_k)
        
        # Check which results are from relevant documents
        retrieved_docs = [chunk['doc_id'] for chunk, score in retrieved]
        relevant_retrieved = [d for d in retrieved_docs if d in expected_docs]
        unique_relevant_found = set(relevant_retrieved)
        
        # Compute metrics
        precision = len(relevant_retrieved) / top_k if top_k > 0 else 0
        recall = len(unique_relevant_found) / len(expected_docs) if expected_docs else 0
        
        # F1 score (harmonic mean of precision and recall)
        if precision + recall > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0
        
        results.append({
            'query': query,
            'description': tc['description'],
            'expected_docs': expected_docs,
            'retrieved_docs': retrieved_docs,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'top_score': retrieved[0][1] if retrieved else 0
        })
    
    return results


# Run evaluation
eval_results = evaluate_retrieval(rag, test_cases, top_k=3)

# Print formatted results
print("=" * 85)
print("RAG RETRIEVAL EVALUATION RESULTS")
print("=" * 85)
print(f"{'Test Case':<35} {'Prec@3':>8} {'Rec@3':>8} {'F1':>8} {'Top Score':>10}")
print("-" * 85)

for r in eval_results:
    print(f"{r['description']:<35} {r['precision']:>8.2f} {r['recall']:>8.2f} "
          f"{r['f1']:>8.2f} {r['top_score']:>10.4f}")

# Averages
avg_precision = np.mean([r['precision'] for r in eval_results])
avg_recall = np.mean([r['recall'] for r in eval_results])
avg_f1 = np.mean([r['f1'] for r in eval_results])
print("-" * 85)
print(f"{'AVERAGE':<35} {avg_precision:>8.2f} {avg_recall:>8.2f} {avg_f1:>8.2f}")
print("\nDetailed retrieval breakdown:")
for r in eval_results:
    print(f"  Q: {r['query'][:50]}...")
    print(f"    Expected: {r['expected_docs']} | Got: {r['retrieved_docs']}")

In [ ]:
# --- Visualization: Precision & Recall per Test Case ---

fig, ax = plt.subplots(figsize=(12, 5))

labels = [r['description'] for r in eval_results]
precisions = [r['precision'] for r in eval_results]
recalls = [r['recall'] for r in eval_results]

x = np.arange(len(labels))
width = 0.35

bars1 = ax.bar(x - width/2, precisions, width, label='Precision@3',
               color='#1976d2', alpha=0.85, edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x + width/2, recalls, width, label='Recall@3',
               color='#ff7043', alpha=0.85, edgecolor='white', linewidth=1.5)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4), textcoords='offset points',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4), textcoords='offset points',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('RAG Retrieval Quality: Precision@3 vs Recall@3 per Test Case',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=9)
ax.set_ylim(0, 1.25)
ax.legend(fontsize=11)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3, label='Perfect')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

## ⚠️ Common Failure Modes & Fixes

Even well-built RAG systems can fail. Here are the **five most common problems** and how to fix them:

| # | Failure Mode | What Happens | Fix |
|---|-------------|-------------|-----|
| 1 | **Vocabulary mismatch** | User says "space rock" but docs say "asteroid" | Add synonyms, use better embeddings |
| 2 | **Chunk too large** | Retrieved chunk has the answer buried in noise | Reduce chunk size, use better splitting |
| 3 | **Chunk too small** | Answer is split across chunks, context is lost | Increase chunk size or overlap |
| 4 | **Wrong document retrieved** | Cosine similarity matches surface words, not meaning | Use semantic embeddings (e.g., sentence transformers) |
| 5 | **Insufficient context** | Top-k is too low, missing relevant information | Increase top-k, use re-ranking |

In [ ]:
# --- Demo: Common Failure Modes and Fixes ---

print("=" * 70)
print("FAILURE MODE DEMOS")
print("=" * 70)

# Failure Mode 1: Vocabulary Mismatch
print("\n--- Failure Mode 1: Vocabulary Mismatch ---")
print("The user uses different words than the documents.\n")

# The mismatch query
mismatch_query = "space rock belt between planets"
mismatch_results = rag.retrieve(mismatch_query, top_k=3)
print(f"Query: '{mismatch_query}'")
print(f"Top result: {mismatch_results[0][0]['title']} "
      f"(score: {mismatch_results[0][1]:.4f})")
print(f"  Text preview: {mismatch_results[0][0]['text'][:100]}...")

# The fix: query expansion with synonyms
print("\nFIX: Query Expansion (add synonyms)")

synonym_map = {
    'space rock': 'asteroid',
    'space rocks': 'asteroids',
    'shooting star': 'meteor',
    'red planet': 'mars',
}

def expand_query(query, synonyms):
    """Expand a query by adding known synonyms."""
    expanded = query.lower()
    additions = []
    for slang, technical in synonyms.items():
        if slang in expanded:
            additions.append(technical)
    if additions:
        expanded = expanded + " " + " ".join(additions)
    return expanded

expanded = expand_query(mismatch_query, synonym_map)
expanded_results = rag.retrieve(expanded, top_k=3)
print(f"Expanded query: '{expanded}'")
print(f"Top result: {expanded_results[0][0]['title']} "
      f"(score: {expanded_results[0][1]:.4f})")
print(f"  Text preview: {expanded_results[0][0]['text'][:100]}...")

score_diff = expanded_results[0][1] - mismatch_results[0][1]
print(f"\nScore improvement: +{score_diff:.4f}")

# Failure Mode 2: Insufficient Context (top_k too low)
print("\n" + "=" * 70)
print("\n--- Failure Mode 2: Insufficient Context (top_k too low) ---")
print("When a question needs info from multiple documents.\n")

multi_doc_query = "Compare the Moon landings with Mars rover missions"

# With k=1 (too few)
results_k1 = rag.retrieve(multi_doc_query, top_k=1)
docs_k1 = set(c['doc_id'] for c, s in results_k1)
print(f"Query: '{multi_doc_query}'")
print(f"\nWith top_k=1: Found docs = {docs_k1}")
print(f"  Only covers {'1 topic' if len(docs_k1) == 1 else 'both topics'} -- MISSING CONTEXT!")

# With k=5 (enough)
results_k5 = rag.retrieve(multi_doc_query, top_k=5)
docs_k5 = set(c['doc_id'] for c, s in results_k5)
print(f"\nFIX: With top_k=5: Found docs = {docs_k5}")
print(f"  Covers {len(docs_k5)} different documents -- much better coverage!")
for c, s in results_k5:
    print(f"    - {c['title']} (chunk {c['chunk_index']}, score: {s:.4f})")

---

## 🏭 Production Considerations

Our pipeline works great for learning, but real-world RAG systems need more. Here's what changes when you go to production:

### Scaling Up
- **Embedding model:** Replace TF-IDF with dense embeddings (e.g., OpenAI `text-embedding-3-small`, Cohere, Sentence-BERT). These understand *meaning*, not just word overlap.
- **Vector store:** Use a dedicated vector database (Pinecone, Weaviate, Qdrant, ChromaDB, FAISS) that can handle millions of vectors with millisecond search.
- **LLM integration:** Connect to GPT-4, Claude, Llama, or another model for actual text generation.

### Improving Quality
- **Re-ranking:** After initial retrieval, use a cross-encoder model to re-rank results for higher precision.
- **Hybrid search:** Combine vector search with keyword search (BM25) for better recall.
- **Metadata filtering:** Filter by date, source, category before vector search.
- **Query rewriting:** Use an LLM to reformulate vague queries before searching.

### Monitoring & Maintenance
- **Logging:** Track queries, retrieved chunks, and user feedback.
- **A/B testing:** Compare different chunk sizes, embedding models, and prompt templates.
- **Index freshness:** Keep your document index updated as source documents change.
- **Guardrails:** Prevent the system from answering questions outside its knowledge domain.

---

## 🎯 Key Takeaways

1. **A RAG pipeline has two phases:** an offline *indexing* phase (load, chunk, embed, store) and an online *query* phase (embed query, search, build prompt, generate).

2. **Each component matters:** Bad chunking means bad retrieval. Bad retrieval means bad answers. The pipeline is only as strong as its weakest link.

3. **Prompt engineering is crucial:** The same retrieved context can produce wildly different answers depending on how you structure the prompt. Use task-specific templates.

4. **Evaluation is not optional:** Always measure precision and recall. Without metrics, you're flying blind.

5. **Common failures are predictable:** Vocabulary mismatch, wrong chunk size, and insufficient context are the top issues. Each has known fixes.

6. **Start simple, iterate:** Our TF-IDF + cosine similarity pipeline is a solid foundation. Upgrade to dense embeddings and real vector stores when you need better performance.

---

## 🧩 Test Your Understanding

<details>
<summary><strong>1. Why do we preprocess text before chunking and embedding?</strong></summary>

Preprocessing normalizes the text so that the same word always looks the same to our system. Without it, "The Moon" and "the moon" would be treated as different tokens. We also remove special characters that add noise without adding meaning. This improves both chunking (cleaner sentence boundaries) and embedding quality (consistent vocabulary).
</details>

<details>
<summary><strong>2. What happens if we set chunk_size too small or too large?</strong></summary>

**Too small:** Individual chunks lose context. A chunk might say "It weighs 6 billion tons" without mentioning what "it" refers to (a neutron star). The answer gets split across multiple chunks that might not all be retrieved.

**Too large:** Chunks contain too much information, so when you retrieve one, the relevant fact is buried in noise. The LLM has to work harder to find the answer, and you might hit token limits faster.

The sweet spot depends on your documents and questions — typically 200-500 characters for factual Q&A.
</details>

<details>
<summary><strong>3. Why is TF-IDF limited compared to dense embeddings for RAG?</strong></summary>

TF-IDF only matches on exact words (lexical matching). If your query says "space rock" but the document says "asteroid," TF-IDF will miss the connection because those are different tokens. Dense embeddings (from models like BERT or sentence transformers) understand that "space rock" and "asteroid" are semantically similar because they've learned word meanings from massive text corpora.
</details>

<details>
<summary><strong>4. Why do we include source metadata with each chunk?</strong></summary>

Metadata serves multiple purposes:
- **Attribution:** The LLM can cite which document the information came from, making answers verifiable.
- **Filtering:** You can filter search results by source, date, or category before ranking.
- **Debugging:** When retrieval goes wrong, metadata helps you trace which document and chunk was retrieved and why.
- **Deduplication:** If the same information appears in multiple documents, metadata helps you avoid showing duplicates.
</details>

<details>
<summary><strong>5. How would you improve this pipeline for production use?</strong></summary>

Key improvements:
1. **Replace TF-IDF** with dense embeddings (e.g., sentence-transformers, OpenAI embeddings) for semantic understanding.
2. **Use a real vector database** (Pinecone, ChromaDB, FAISS) that can scale to millions of documents with fast approximate nearest neighbor search.
3. **Add re-ranking** with a cross-encoder model to improve precision after initial retrieval.
4. **Connect to an LLM** (GPT-4, Claude) for actual answer generation instead of just building prompts.
5. **Implement hybrid search** combining BM25 keyword search with vector search for better recall.
6. **Add monitoring** to track retrieval quality, latency, and user satisfaction over time.
</details>

---

## 🚀 What's Next?

Now that you can build a complete RAG pipeline from scratch, it's time to make it **smarter**. In the next notebook, we'll explore **Agentic RAG** — where the system can reason about *what* to retrieve, *when* to retrieve it, and *how* to combine information from multiple steps.

**Next:** [06 - Agentic RAG with LangGraph](06_agentic_rag_langgraph.ipynb)

Topics we'll cover:
- Using an LLM as a "reasoning engine" that decides when to search
- Multi-step retrieval (search, read, search again)
- Tool use and function calling in RAG
- Building a RAG agent with LangGraph

---

*Notebook 05 of the RAG series. Built with only numpy, matplotlib, math, re, and collections.*